In [1]:
# Config & imports

import os
import numpy as np
import pandas as pd

# Modeling
from xgboost import XGBClassifier
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold,
    cross_val_predict, cross_val_score
)

# Metrics & plots
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_recall_fscore_support
)
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# I/O
DATA_PATH = "./data/train_mp.csv"
os.makedirs("./visual", exist_ok=True)

# Excluded feature columns
NON_FEATURE_COLS = {
    "material_id", "formula", "composition", "composition_obj",
    "formation_energy_per_atom", "band_gap"
}

# Class names for plots
CLASS_NAMES = ["C", "SC"]  # 0, 1


In [2]:
# Load data & define conductor/semiconductor classes

df = pd.read_csv(DATA_PATH)

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
X_all = df[feature_cols].select_dtypes(include=[np.number]).copy()

# Binary labels from band_gap
bg = pd.to_numeric(df["band_gap"], errors="coerce")
tol = 1e-8
y_all = pd.Series(np.nan, index=bg.index, dtype="float64")

y_all[(bg >= 0.1) & (bg <= 3.0)] = 1
y_all[np.abs(bg - 0.0) <= tol] = 0

mask_valid = y_all.notna()
dropped = (~mask_valid).sum()
print(f"Total rows: {len(df)} | Kept (classified): {mask_valid.sum()} | Dropped (outside spec): {dropped}")

X_all = X_all.loc[mask_valid].reset_index(drop=True)
y_all = y_all.loc[mask_valid].astype(int).reset_index(drop=True)

print("Class counts:", dict(zip(*np.unique(y_all, return_counts=True))))
print("n_features (numeric):", X_all.shape[1])
print("first 5 feature columns:", feature_cols[:5])


Total rows: 6419 | Kept (classified): 5584 | Dropped (outside spec): 835
Class counts: {np.int64(0): np.int64(3340), np.int64(1): np.int64(2244)}
n_features (numeric): 250
first 5 feature columns: ['H', 'He', 'Li', 'Be', 'B']


In [3]:
# Train/test split & imputation

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_all
)

# Median imputation
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test  = X_test.fillna(X_train.median(numeric_only=True))

print("train shape:", X_train.shape, "| test shape:", X_test.shape)
print("train class counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("test  class counts:", dict(zip(*np.unique(y_test, return_counts=True))))


train shape: (4467, 250) | test shape: (1117, 250)
train class counts: {np.int64(0): np.int64(2672), np.int64(1): np.int64(1795)}
test  class counts: {np.int64(0): np.int64(668), np.int64(1): np.int64(449)}


In [4]:
# XGBoost classifier + GridSearchCV

TREE_METHOD = "hist"  # or "gpu_hist" if CUDA GPU available

xgb = XGBClassifier(
    random_state=RANDOM_STATE,
    tree_method=TREE_METHOD,
    use_label_encoder=False,
    eval_metric="logloss",
    n_jobs=1,
    scale_pos_weight=1.0,  # handle imbalance manually if needed
)

param_grid = {
    "n_estimators": [400, 800],
    "max_depth": [6, 10],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3],
    "gamma": [0.0, 0.2],
    "reg_lambda": [1.0, 5.0],
    "reg_alpha": [0.0, 0.5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

gscv = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

gscv.fit(X_train, y_train)
best_clf = gscv.best_estimator_
print("best params:", gscv.best_params_)
print("best cv F1_macro:", gscv.best_score_)


Fitting 5 folds for each of 512 candidates, totalling 2560 fits


/home/adroit/miniconda3/envs/material/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:26:54] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/adroit/miniconda3/envs/material/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:26:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/adroit/miniconda3/envs/material/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:26:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/adroit/miniconda3/envs/material/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:26:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fo

best params: {'colsample_bytree': 1.0, 'gamma': 0.0, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'n_estimators': 400, 'reg_alpha': 0.0, 'reg_lambda': 5.0, 'subsample': 0.8}
best cv F1_macro: 0.8595423024422827


In [5]:
# Predict on new data 

new_features_path = "./data/predict_800.csv"          # input file
out_path = "./data/pred_band_gap.csv"                 # output file

new_df = pd.read_csv(new_features_path)

# Keep a copy of the mxene column before dropping
if "mxene" not in new_df.columns:
    raise ValueError("Input file must contain a 'mxene' column to identify entries.")
mxene_col = new_df["mxene"].copy()

# Drop non-feature columns (same as training exclusion list)
NON_FEATURE_COLS = {"mxene", "composition_obj"}
new_df = new_df.drop(columns=[c for c in new_df.columns if c in NON_FEATURE_COLS],
                     errors="ignore")

# Ensure same feature order as training
missing = set(X_train.columns) - set(new_df.columns)
extra   = set(new_df.columns) - set(X_train.columns)

if missing:
    raise ValueError(f"The following required feature columns are missing in the input: {sorted(list(missing))[:10]} ...")

# Reindex to match training feature order
X_new = new_df.reindex(columns=X_train.columns, fill_value=np.nan)

# Fill missing values with training medians
X_new = X_new.fillna(X_train.median(numeric_only=True))

# Predict classes (0 = conductor, 1 = semiconductor)
pred_cls = best_clf.predict(X_new)
pred_name = pd.Series(pred_cls).map({0: "conductor", 1: "semiconductor"})

# Combine results and save (only mxene + prediction columns)
out = pd.DataFrame({
    "mxene": mxene_col,
    "predicted_class": pred_cls,
    "predicted_label": pred_name
})

out.to_csv(out_path, index=False)
print(f"Saved predictions to: {out_path}")
out.head()


Saved predictions to: ./data/pred_band_gap.csv


,mxene,predicted_class,predicted_label
0,Cr2CBr2,0,conductor
1,Hf2CBr2,1,semiconductor
2,Mo2CBr2,0,conductor
3,Nb2CBr2,0,conductor
4,Sc2CBr2,1,semiconductor
